# 03 -- Normalization into Stage 3 Domain Models

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Shows raw, provider-shaped records (`normalization.Raw*Record`) being converted into the platform's *existing* Stage 3 domain models (`FilingFundamentals`/`DailyPriceObservation`/`UniverseMembershipRecord`/`SectorRecord`) -- never a second, parallel domain model. A record that fails to normalize is rejected and reported, never silently coerced.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from atlas_quant.strategies.filing_momentum_ml.production.normalization import (
    RawFilingRecord, normalize_filings,
)
from datetime import date, datetime

good = _fixtures.build_raw_filings()[0]
bad = RawFilingRecord(
    symbol="ZZZ", asset_class="not_a_real_asset_class", fiscal_period="Q1", fiscal_year=2023,
    quarter_end=date(2023, 3, 31), filed_at=datetime(2023, 4, 30), revenue=None, gross_profit=None,
    operating_income=None, net_income=None, diluted_eps=None, stockholders_equity=None,
    operating_cash_flow=None, capital_expenditure=None, accession_number=None,
    source="synthetic_fixture", retrieved_at=datetime(2023, 6, 1),
)
normalized, issues = normalize_filings([good, bad])
print(f"{len(normalized)} normalized, {len(issues)} rejected")
print("normalized type:", type(normalized[0]).__name__)
for issue in issues:
    print(f"  [{issue.severity.value}] {issue.subject}: {issue.message}")


## Findings

- The malformed record (`asset_class="not_a_real_asset_class"`) is dropped and reported at `error` severity -- it never crashes the batch and never falls back to a default asset class.

## Limitations

- Only asset-class rejection is demonstrated here; the same reject-and-report pattern applies to prices/universe/sector records and to any Stage 3 `__post_init__` validation failure (e.g. a negative price).